In [1]:
import xml.etree.ElementTree as ET
import pandas as pd
from pathlib import Path
import numpy as np

In [2]:
RAW_XML_PATH = Path("../data/export.xml")
PROCESSED_PATH = Path("../data/processed.csv")

In [3]:
# --- 1. Parse XML
tree = ET.parse(RAW_XML_PATH)
root = tree.getroot()

attrs = set()
for i, record in enumerate(root.findall('Record')):
    attrs.update(record.attrib.keys())
    if i > 1000:
        break
print(attrs)


{'sourceVersion', 'unit', 'endDate', 'type', 'startDate', 'creationDate', 'sourceName', 'value'}


In [4]:
root

<Element 'HealthData' at 0x113168d10>

In [5]:
records = []
for record in root.findall('Record'):
    rtype = record.get('type')
    start = record.get('startDate')
    end = record.get('endDate')
    value = record.get('value')
    unit = record.get('unit')
    source = record.get('sourceName')
    records.append((rtype, start, end, value, unit, source))


In [6]:
df_all = pd.DataFrame(records, columns=["rtype","start","end","value","unit","source"])
df_all['rtype'] = df_all['rtype'].str.split("Identifier").str[-1]
df_all['rtype'].unique()

array(['BloodGlucose', 'DietaryWater', 'BodyMassIndex', 'Height',
       'BodyMass', 'HeartRate', 'OxygenSaturation', 'RespiratoryRate',
       'BodyFatPercentage', 'LeanBodyMass', 'StepCount',
       'DistanceWalkingRunning', 'BasalEnergyBurned',
       'ActiveEnergyBurned', 'FlightsClimbed', 'DietaryFatTotal',
       'DietaryFatPolyunsaturated', 'DietaryFatMonounsaturated',
       'DietaryFatSaturated', 'DietaryCholesterol', 'DietarySodium',
       'DietaryCarbohydrates', 'DietaryFiber', 'DietarySugar',
       'DietaryEnergyConsumed', 'DietaryProtein', 'DietaryVitaminA',
       'DietaryVitaminB6', 'DietaryVitaminB12', 'DietaryVitaminC',
       'DietaryVitaminD', 'DietaryVitaminE', 'DietaryVitaminK',
       'DietaryCalcium', 'DietaryIron', 'DietaryThiamin',
       'DietaryRiboflavin', 'DietaryNiacin', 'DietaryFolate',
       'DietaryBiotin', 'DietaryPantothenicAcid', 'DietaryPhosphorus',
       'DietaryIodine', 'DietaryMagnesium', 'DietaryZinc',
       'DietarySelenium', 'DietaryCoppe

In [7]:
df_all['source'].unique()

array(['Lingo', 'Bevel', 'Health', 'VeSync', 'Kay’s Apple\xa0Watch',
       'OKK', '薄荷健康', 'MyFitnessPal', 'Cal AI', 'Kay’s iPhone',
       'Blood Oxygen', 'Kay14', 'iPhone', 'GymKit', 'Nike Run Club',
       'StepsApp', 'Sleep', 'Sleep Cycle', 'Cycle Tracking', '番茄ToDo'],
      dtype=object)

In [8]:
df_all[df_all['source']=='Sleep Cycle']

,rtype,start,end,value,unit,source
3184255,SleepAnalysis,2020-06-09 22:37:33 -0400,2020-06-10 05:47:05 -0400,HKCategoryValueSleepAnalysisInBed,None,Sleep Cycle
3184256,SleepAnalysis,2020-06-09 22:56:10 -0400,2020-06-10 05:47:05 -0400,HKCategoryValueSleepAnalysisAsleepUnspecified,None,Sleep Cycle
3184257,SleepAnalysis,2020-06-10 23:32:49 -0400,2020-06-11 03:32:09 -0400,HKCategoryValueSleepAnalysisAsleepUnspecified,None,Sleep Cycle
3184258,SleepAnalysis,2020-06-11 03:52:26 -0400,2020-06-11 05:56:49 -0400,HKCategoryValueSleepAnalysisAsleepUnspecified,None,Sleep Cycle
3184259,SleepAnalysis,2020-06-10 23:24:42 -0400,2020-06-11 06:10:21 -0400,HKCategoryValueSleepAnalysisInBed,None,Sleep Cycle
...,...,...,...,...,...,...
3188454,SleepAnalysis,2023-10-23 22:07:11 -0400,2023-10-24 06:13:41 -0400,HKCategoryValueSleepAnalysisInBed,None,Sleep Cycle
3188692,SleepAnalysis,2023-11-01 22:40:44 -0400,2023-11-02 02:35:02 -0400,HKCategoryValueSleepAnalysisAsleepUnspecified,None,Sleep Cycle
3188693,SleepAnalysis,2023-11-02 03:07:52 -0400,2023-11-02 04:32:56 -0400,HKCategoryValueSleepAnalysisAsleepUnspecified,None,Sleep Cycle
3188694,SleepAnalysis,2023-11-02 05:02:47 -0400,2023-11-02 05:49:03 -0400,HKCategoryValueSleepAnalysisAsleepUnspecified,None,Sleep Cycle


In [9]:
df_all[df_all['rtype']=='SleepAnalysis']['source'].unique()

array(['Sleep Cycle', 'Kay’s Apple\xa0Watch', 'Kay14'], dtype=object)

In [ ]:
df_all['start'] = pd.to_datetime(df_all['start'], errors='coerce')
df_all['end'] = pd.to_datetime(df_all['end'], errors='coerce')

In [ ]:
w1_start, w1_end = pd.Timestamp("2025-07-28", tz="UTC-04:00"), pd.Timestamp("2025-07-28", tz="UTC-04:00") + pd.Timedelta(days=14)
w2_start, w2_end = pd.Timestamp("2026-03-25", tz="UTC-04:00"), pd.Timestamp("2026-03-25", tz="UTC-04:00") + pd.Timedelta(days=14)
df = df_all[
    (((df_all['start'] >= w1_start) & (df_all['start'] <= w1_end)) |
     ((df_all['start'] >= w2_start) & (df_all['start'] <= w2_end)))
].copy()

df['window'] = np.where(df['start'] <= w1_end, 1, 2)

In [ ]:
sources_to_keep = ['Lingo', 'Bevel', 'Kay\u2019s Apple\xa0Watch']
variables_to_keep = ['BloodGlucose', 'HeartRate','StepCount','ActiveEnergyBurned',
           'DietaryCarbohydrates','DietaryProtein','DietaryFatTotal',
           'DietaryFiber','DietaryEnergyConsumed','SleepAnalysis']
df = df[df['rtype'].isin(variables_to_keep) & df['source'].isin(sources_to_keep)]

In [ ]:
df_sleep = df[df['rtype'] == 'SleepAnalysis'].copy()
df = df[df['rtype'] != 'SleepAnalysis'].copy()

df['start'] = pd.to_datetime(df['start'], errors='coerce')
df['end'] = pd.to_datetime(df['end'], errors='coerce')
df['value'] = pd.to_numeric(df['value'], errors='coerce')
df = df.dropna(subset=['value'])

df_sleep['start'] = pd.to_datetime(df_sleep['start'], errors='coerce')
df_sleep['end'] = pd.to_datetime(df_sleep['end'], errors='coerce')
# value stays as string category — don't coerce to numeric

In [ ]:
food = df[df['rtype'] == 'DietaryEnergyConsumed'].sort_values('start').copy()
gap = food['start'].diff().dt.total_seconds().div(60).fillna(999)
food['meal_id'] = (gap > 15).cumsum()

meals = food.groupby('meal_id').agg(
    meal_start=('start', 'min'),
    meal_end=('end', 'max'),
    total_energy=('value', 'sum'),
    n_entries=('value', 'count'),
    window=('window', 'first')
).reset_index()

macros = ['DietaryEnergyConsumed', 'DietaryProtein', 'DietaryCarbohydrates', 'DietaryFatTotal', 'DietaryFiber']
wide = df[df['rtype'].isin(macros)].pivot_table(index='start', columns='rtype', values='value', aggfunc='sum').reset_index()
wide = pd.merge_asof(wide.sort_values('start'), food[['start', 'meal_id']].sort_values('start'), on='start', direction='nearest', tolerance=pd.Timedelta('1min'))

meal_dataset = wide.groupby('meal_id').sum(numeric_only=True).reset_index()
meal_dataset = meal_dataset.merge(meals[['meal_id', 'meal_start', 'meal_end', 'window']], on='meal_id', how='left')

In [ ]:
glucose = df[df['rtype'] == 'BloodGlucose'][['start', 'value']].sort_values('start').rename(columns={'value': 'glucose'})

def glucose_features(meal_start, glucose_df, baseline_min=15, response_hr=2):
    baseline_window = glucose_df[(glucose_df['start'] >= meal_start - pd.Timedelta(minutes=baseline_min)) & (glucose_df['start'] < meal_start)]
    response_window = glucose_df[(glucose_df['start'] >= meal_start) & (glucose_df['start'] <= meal_start + pd.Timedelta(hours=response_hr))]

    if baseline_window.empty or response_window.empty:
        return pd.Series({'baseline_glucose': np.nan, 'peak_rise': np.nan, 'iauc': np.nan,
                           'n_baseline_pts': len(baseline_window), 'n_response_pts': len(response_window)})

    baseline = baseline_window['glucose'].mean()
    peak_rise = response_window['glucose'].max() - baseline

    t = (response_window['start'] - meal_start).dt.total_seconds() / 60
    g_clipped = (response_window['glucose'] - baseline).clip(lower=0)
    iauc = np.trapezoid(g_clipped, t)

    return pd.Series({'baseline_glucose': baseline, 'peak_rise': peak_rise, 'iauc': iauc,
                       'n_baseline_pts': len(baseline_window), 'n_response_pts': len(response_window)})

feats = meal_dataset['meal_start'].apply(lambda t: glucose_features(t, glucose))
meal_dataset = pd.concat([meal_dataset, feats], axis=1)

In [ ]:
print(meal_dataset[['n_baseline_pts', 'n_response_pts']].describe())
print((meal_dataset['n_baseline_pts'] == 0).sum(), (meal_dataset['n_response_pts'] == 0).sum())

       n_baseline_pts  n_response_pts
count       91.000000       91.000000
mean         2.967033       23.956044
std          0.314485        0.419314
min          0.000000       20.000000
25%          3.000000       24.000000
50%          3.000000       24.000000
75%          3.000000       24.000000
max          3.000000       24.000000
1 0


In [ ]:
meal_dataset = meal_dataset[meal_dataset['n_baseline_pts'] > 0].copy()
print(meal_dataset.shape)  # expect (46, ...)

(90, 14)


In [ ]:
print(meal_dataset['window'].value_counts())

window
2    46
1    44
Name: count, dtype: int64


In [ ]:
activity = df[df['rtype'].isin(['StepCount', 'HeartRate', 'ActiveEnergyBurned'])][['rtype', 'start', 'end', 'value']].copy()

def activity_features(meal_start, activity_df, pre_min=60, post_min=60):
    pre = activity_df[(activity_df['start'] >= meal_start - pd.Timedelta(minutes=pre_min)) & (activity_df['start'] < meal_start)]
    post = activity_df[(activity_df['start'] > meal_start) & (activity_df['start'] <= meal_start + pd.Timedelta(minutes=post_min))]

    def agg(window):
        steps = window[window['rtype'] == 'StepCount']['value'].sum()
        active_energy = window[window['rtype'] == 'ActiveEnergyBurned']['value'].sum()
        hr_mean = window[window['rtype'] == 'HeartRate']['value'].mean()
        return steps, active_energy, hr_mean

    pre_steps, pre_energy, pre_hr = agg(pre)
    post_steps, post_energy, post_hr = agg(post)

    return pd.Series({
        'pre_steps': pre_steps, 'pre_active_energy': pre_energy, 'pre_hr': pre_hr,
        'post_steps': post_steps, 'post_active_energy': post_energy, 'post_hr': post_hr,
        'walked_after_eating': int(post_steps > 200)  # threshold worth eyeballing against your actual data distribution
    })

act_feats = meal_dataset['meal_start'].apply(lambda t: activity_features(t, activity))
meal_dataset = pd.concat([meal_dataset, act_feats], axis=1)

In [ ]:
sleep = df[df['rtype'] == 'SleepAnalysis'][['start', 'end', 'value']].copy()
asleep_categories = [v for v in sleep['value'].unique() if 'Asleep' in v]  # excludes Awake/InBed

def sleep_hours_before(meal_start, sleep_df, lookback_hr=18):
    window = sleep_df[(sleep_df['end'] <= meal_start) & (sleep_df['end'] >= meal_start - pd.Timedelta(hours=lookback_hr)) & (sleep_df['value'].isin(asleep_categories))]
    total_sec = (window['end'] - window['start']).dt.total_seconds().sum()
    return total_sec / 3600

meal_dataset['hours_slept'] = meal_dataset['meal_start'].apply(lambda t: sleep_hours_before(t, sleep))

In [ ]:
meal_dataset.to_parquet('../data/meal_dataset.parquet')
glucose.to_parquet('../data/glucose_timeseries.parquet')


In [ ]:
# import matplotlib.pyplot as plt

# # Pick a single day that you know has glucose & lifestyle data
# day = "2025-08-01"  # <-- change this to a date you have in your dataset

# day_df = wide_5min.loc[day]

# fig, ax1 = plt.subplots(figsize=(12,6))

# # --- Plot glucose on the left y-axis ---
# ax1.plot(day_df.index, day_df['glucose_mmol'], color='tab:red', marker='o', linestyle='-', label='Glucose (mmol/L)')
# ax1.set_ylabel('Glucose (mmol/L)', color='tab:red')
# ax1.tick_params(axis='y', labelcolor='tab:red')

# # --- Create second y-axis for carbs/steps ---
# ax2 = ax1.twinx()
# # ax2.bar(day_df.index, day_df['carbs_g'], width=0.003, color='tab:blue', alpha=0.5, label='Carbs (g)')
# # ax2.plot(day_df.index, day_df['steps'], color='tab:green', alpha=0.7, label='Steps')
# ax2.plot(day_df.index, day_df['active_kcal'], color='tab:green', alpha=0.7, label='Cal Burned')
# ax2.plot(day_df.index, day_df['kcal'], color='tab:blue', alpha=0.7, label='Cal Intake')
# ax2.set_ylabel('Carbs (g) / Steps', color='tab:blue')
# ax2.tick_params(axis='y', labelcolor='tab:blue')

# # --- Format ---
# fig.suptitle(f"Daily CGM + Lifestyle Features: {day}", fontsize=14)
# fig.autofmt_xdate()
# fig.legend(loc='upper left', bbox_to_anchor=(0.1, 0.9))

# plt.tight_layout()
# plt.show()